In [1]:
import subprocess
import os
import time

# Get the kernel PID
mgr_parent_pid = os.getpid()
print(f"Kernel PID: {mgr_parent_pid}")

# Define the strace command
strace_cmd = [
    "sudo", "-S",  # -S reads password from stdin
    "strace",
    "-qqq",  
    "-r",   
    "-z",  
    "-f",    # Follow forks/threads
    "-o", "/home/admin/shahadat/floability-env.cli/strace_manager.txt", 
    "-p", str(mgr_parent_pid),
]

# Start strace in the background with sudo
mgr_strace = subprocess.Popen(
    strace_cmd,
    stdin=subprocess.PIPE,  # Enable stdin for password
    stdout=subprocess.PIPE,  
    stderr=subprocess.PIPE, 
    preexec_fn=os.setsid
)

# Send the sudo password
password = "cdm_depaul"  # Replace with your actual password
mgr_strace.stdin.write(password.encode() + b"\n")  # Encode and add newline
mgr_strace.stdin.flush() 

print("strace is running in the background. Execute your cells now!")

time.sleep(1)
mgr_strace.stdin.close()

from functools import wraps
def worker(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with open('_tmp.txt', 'w') as f: 
            f.write('hello1')
            result = func(*args, **kwargs)
            f.write('hello2')
            print(result)
            return result
    return wrapper


Kernel PID: 205199
strace is running in the background. Execute your cells now!


In [2]:
import os
from coffea.nanoevents import NanoEventsFactory
from coffea import processor

import dask_awkward as dak
import hist.dask as hda

import warnings
warnings.filterwarnings("error", module="coffea.*")


use_taskvine = True

if use_taskvine:
    from ndcctools.taskvine import DaskVine

    vine_manager = DaskVine(
        [9123, 9128],
        # name=f"{os.environ.get("VINE_MANAGER_NAME")}",
    )

    executor_args = {
        "scheduler": vine_manager,
        "worker_transfers": True,
        # "task_mode": "function-calls",
    }
else:
    from distributed import Client
    client = Client()

    executor_args = {}

/home/admin/miniconda3/envs/my_coffea_env/lib/python3.11/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [3]:
# data_abs_path = os.path.abspath("data/small_data.root")
# data_url = f"file://{data_abs_path}"

# data_file = (data_url,)

# from coffea.nanoevents.schemas import NanoAODSchema


# NanoAODSchema.warn_missing_crossrefs = False


# events = NanoEventsFactory.from_root(
#     {data_file: "/Events"},
#     metadata={"dataset": "SingleMu"}
# ).events()

# q1_hist = (
#     hda.Hist.new.Reg(100, 0, 200, name="met", label="$E_{T}^{miss}$ [GeV]")
#     .Double()
#     .fill(events.MET.pt)
# )

# q1_hist.compute(**executor_args).plot1d()

# dak.necessary_columns(q1_hist)

In [4]:
# Code generated by ChatGPT to control the run time

import threading
from IPython.display import display, Javascript

def long_task():
    events = NanoEventsFactory.from_root(
        {data_file: "/Events"},
        metadata={"dataset": "SingleMu"}
    ).events()

    q1_hist = (
        hda.Hist.new.Reg(100, 0, 200, name="met", label="$E_{T}^{miss}$ [GeV]")
        .Double()
        .fill(events.MET.pt)
    )

    q1_hist.compute(**executor_args).plot1d()
    dak.necessary_columns(q1_hist)

def run_with_timeout(timeout=120):
    thread = threading.Thread(target=long_task)
    thread.start()
    thread.join(timeout)
    if thread.is_alive():
        print(f"⏰ Timed out after {timeout} seconds. Moving to the next cell.")
        display(Javascript('Jupyter.notebook.execute_cell_range(Jupyter.notebook.get_selected_index()+1, Jupyter.notebook.get_selected_index()+2)'))
    else:
        print("✅ Task completed.")

run_with_timeout()


Exception in thread Thread-5 (long_task):
Traceback (most recent call last):
  File "/home/admin/miniconda3/envs/my_coffea_env/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/admin/miniconda3/envs/my_coffea_env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/home/admin/miniconda3/envs/my_coffea_env/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_205199/718994875.py", line 8, in long_task
NameError: name 'data_file' is not defined


✅ Task completed.


In [5]:
import signal

try:
    # Terminate the process group (sudo and strace)
    os.killpg(os.getpgid(mgr_strace.pid), signal.SIGTERM)
    mgr_strace.wait(timeout=5)
    print(f"strace group terminated, exit code: {mgr_strace.returncode}")
except subprocess.TimeoutExpired:
    print("Graceful termination timed out, forcing kill")
    # Kill the process group
    os.killpg(os.getpgid(mgr_strace.pid), signal.SIGKILL)
    mgr_strace.wait()
    # Double-check strace is gone
    #subprocess.run(f"sudo pkill -f 'strace -p {mgr_parent_pid}'", shell=True)
    print("strace group killed")


strace group terminated, exit code: -15
